# 🚀 MLOps Essentials: Deployment & Production ML

**Learning Objectives:**
- Understand CI/CD pipelines for ML models
- Learn Docker containerization for ML applications
- Implement model drift monitoring in production

---


## Part 1: CI/CD for Machine Learning Models

### What is CI/CD in MLOps?

**CI (Continuous Integration):** Automatically test and validate code/model changes

**CD (Continuous Deployment):** Automatically deploy validated models to production

### Key Differences: Traditional Software vs ML CI/CD

| Aspect | Traditional CI/CD | ML CI/CD |
|--------|------------------|----------|
| Testing | Code tests | Code + Data + Model tests |
| Artifacts | Binaries | Models + Datasets + Configs |
| Versioning | Code only | Code + Data + Models |
| Triggers | Code changes | Code/Data/Model performance changes |

In [1]:
# Install required libraries
!pip install -q scikit-learn mlflow pandas numpy joblib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.2/779.2 kB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 19.6 MB/s eta 0:00:00


### Step 1: Create a Sample ML Model

In [2]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib
import json
from datetime import datetime

In [3]:
# Load dataset
iris = load_iris()
X, y = iris.data, iris.target

In [4]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [6]:
# Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

In [7]:
print(f"Model Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

Model Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



### Step 2: Model Versioning & Registry

In [8]:
# Model versioning metadata
model_metadata = {
    "model_name": "iris_classifier",
    "version": "v1.0.0",
    "timestamp": datetime.now().isoformat(),
    "accuracy": float(accuracy),
    "hyperparameters": {
        "n_estimators": 100,
        "random_state": 42
    },
    "training_samples": len(X_train),
    "features": iris.feature_names
}

# Save model and metadata
joblib.dump(model, 'iris_model_v1.pkl')
with open('model_metadata.json', 'w') as f:
    json.dump(model_metadata, f, indent=2)

print("Model saved with versioning")
print(json.dumps(model_metadata, indent=2))

Model saved with versioning
{
  "model_name": "iris_classifier",
  "version": "v1.0.0",
  "timestamp": "2026-01-07T16:00:18.592971",
  "accuracy": 1.0,
  "hyperparameters": {
    "n_estimators": 100,
    "random_state": 42
  },
  "training_samples": 120,
  "features": [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)"
  ]
}


## Model Creation and Metadata

Code Explanation: Before deploying anything, we need a model. This code trains a standard Random Forest classifier on the Iris dataset. Crucially, it creates a model_metadata.json file.

 In MLOps, a model is useless without its context (who trained it, when, what features it uses, and its version).

Key Concept: Model Versioning. We don't just save model.pkl; we save it as v1.0.0 with a timestamp.

### Step 3: Automated Testing Pipeline

In [9]:
# Automated testing for ML models - part of CI pipeline

def run_model_tests(model, X_test, y_test, min_accuracy=0.85):

    # Create result container
    test_results = {"passed": [], "failed": []}

    # Test 1: Model can make predictions
    try:
        predictions = model.predict(X_test)
        test_results["passed"].append("✅ Prediction Test")
    except Exception as e:
        test_results["failed"].append(f"❌ Prediction Test: {str(e)}")
        return test_results

    # Test 2: Accuracy threshold
    accuracy = accuracy_score(y_test, predictions)
    if accuracy >= min_accuracy:
        test_results["passed"].append(f"✅ Accuracy Test: {accuracy:.4f} >= {min_accuracy}")
    else:
        test_results["failed"].append(f"❌ Accuracy Test: {accuracy:.4f} < {min_accuracy}")

    # Test 3: Output shape validation
    # Did the model return one prediction per input row?
    # If X_test has 30 rows
    # predictions must have 30 outputs
    if predictions.shape[0] == X_test.shape[0]:
        test_results["passed"].append("✅ Output Shape Test")
    else:
        test_results["failed"].append("❌ Output Shape Test")

    # Test 4: No NaN predictions
    # Are there any NaN (Not a Number) predictions?
    if not np.isnan(predictions).any():
        test_results["passed"].append("✅ NaN Check Test")
    else:
        test_results["failed"].append("❌ NaN Check Test")

    # Test 5: Valid class labels
    # Is the model predicting only known classes?
    unique_preds = np.unique(predictions)
    if all(pred in np.unique(y_test) for pred in unique_preds):
        test_results["passed"].append("✅ Valid Labels Test")
    else:
        test_results["failed"].append("❌ Valid Labels Test")

    return test_results

In [10]:
# Run automated tests
print("Running CI Pipeline Tests...\n")
results = run_model_tests(model, X_test, y_test)

print("Passed Tests:")
for test in results["passed"]:
    print(f"  {test}")

if results["failed"]:
    print("\nFailed Tests:")
    for test in results["failed"]:
        print(f"  {test}")
else:
    print("\n🎉 All tests passed! Model ready for deployment.")

Running CI Pipeline Tests...

Passed Tests:
  ✅ Prediction Test
  ✅ Accuracy Test: 1.0000 >= 0.85
  ✅ Output Shape Test
  ✅ NaN Check Test
  ✅ Valid Labels Test

🎉 All tests passed! Model ready for deployment.


## Automated Testing Pipeline

Code Explanation: In traditional software, we test if code crashes.

 In ML, we must also test if the model is "smart" enough. The run_model_tests function acts as a Quality Gate. It checks:

* Inference: Does the model actually produce an answer?

* Accuracy: Is the performance above our minimum requirement (e.g.,  85%)?

* Sanity: Are there NaN (null) values or impossible labels in the output?

### Real Companies Add Even More Tests

Latency test (must be < 100ms)

Memory test

Bias test

Stability test

Schema test

### Step 4: Deployment Strategies

In [11]:
def blue_green_deployment_simulation():
    """
    Simulate Blue-Green Deployment Strategy
    """
    print("🔵 Blue-Green Deployment Simulation\n")
    print("Step 1: Blue Environment (Current Production) - Model v1.0")
    print("  - Serving 100% of traffic")
    print("  - Accuracy: 96.67%\n")

    print("Step 2: Deploy Green Environment (New Version) - Model v2.0")
    print("  - Parallel deployment")
    print("  - Running health checks...")
    print("  - Accuracy: 98.33%\n")

    print("Step 3: Switch Traffic")
    print("  - Route 100% traffic to Green")
    print("  - Monitor for 5 minutes...\n")

    print("Step 4: Rollback Plan Ready")
    print("  - Blue environment on standby")
    print("  - Can instantly rollback if issues detected\n")

    print("✅ Deployment successful! Green is now production.")



In [ ]:
# Run simulations
blue_green_deployment_simulation()


In [ ]:
def canary_deployment_simulation():
    """
    Simulate Canary Release Strategy
    """
    print("Canary Deployment Simulation\n")

    stages = [
        (5, "Initial canary"),
        (25, "Expanded testing"),
        (50, "Half traffic"),
        (100, "Full rollout")
    ]

    for percentage, stage in stages:
        print(f"Stage: {stage}")
        print(f"  - New model serving: {percentage}% of traffic")
        print(f"  - Monitoring metrics...")
        print(f"  - ✅ No issues detected\n")

    print("🎉 Canary deployment completed successfully!")


In [ ]:
canary_deployment_simulation()

## Deployment Strategies (Simulations)
Code Explanation: These functions explain how we switch from an old model to a new one:

**Blue-Green:** You have two identical environments. You keep the old one (Blue) running while testing the new one (Green). Once Green is ready, you flip a switch.

**Canary:** You "leak" the new model to 5% of users first. If they don't complain and the model performs well, you slowly increase it to 100%.

---

## Part 2: Containerizing ML Models with Docker

### Why Docker for ML?
- **Reproducibility:** Same environment everywhere
- **Isolation:** Dependencies don't conflict
- **Portability:** Run anywhere Docker is installed
- **Scalability:** Easy to orchestrate with Kubernetes

### Step 1: Create Flask API for Model Serving

In [12]:
# Create app.py file
app_code = '''
from flask import Flask, request, jsonify
import joblib
import numpy as np
import json

app = Flask(__name__)

# Load model at startup
model = joblib.load('iris_model_v1.pkl')
with open('model_metadata.json', 'r') as f:
    metadata = json.load(f)

@app.route('/health', methods=['GET'])
def health_check():
    return jsonify({"status": "healthy", "version": metadata["version"]})

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        features = np.array(data['features']).reshape(1, -1)

        prediction = model.predict(features)
        probability = model.predict_proba(features)

        return jsonify({
            "prediction": int(prediction[0]),
            "probabilities": probability[0].tolist(),
            "model_version": metadata["version"]
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 400

@app.route('/info', methods=['GET'])
def model_info():
    return jsonify(metadata)

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print("Flask API created (app.py)")

Flask API created (app.py)


### Step 2: Create Dockerfile

In [ ]:
dockerfile_content = '''
# Use official Python runtime as base image
FROM python:3.9-slim

# Set working directory
WORKDIR /app

# Copy requirements first (for better caching)
COPY requirements.txt .

# Install dependencies
RUN pip install --no-cache-dir -r requirements.txt

# Copy application files
COPY app.py .
COPY iris_model_v1.pkl .
COPY model_metadata.json .

# Expose port
EXPOSE 5000

# Health check
HEALTHCHECK --interval=30s --timeout=3s --start-period=5s --retries=3 \\
  CMD curl -f http://localhost:5000/health || exit 1

# Run application
CMD ["python", "app.py"]
'''

with open('Dockerfile', 'w') as f:
    f.write(dockerfile_content)

print("Dockerfile created")

## Creating the Flask API and Dockerfile
Code Explanation: To make a model accessible to the world, we wrap it in a Web API (using Flask).

**app.py:** Defines the "routes."

*/predict* is where the user sends data to get a result.

Dockerfile: This is a "recipe" for a virtual computer.

 It tells Docker to take Python 3.9, install our libraries, and copy our model into it. This ensures the model runs the same on your laptop, a server, or the cloud.

### Step 3: Create Requirements File

In [13]:
requirements = '''
flask==3.0.0
scikit-learn==1.3.2
numpy==1.24.3
joblib==1.3.2
'''

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print("requirements.txt created")

requirements.txt created


### Step 4: Docker Commands Reference

```bash
# Build Docker image
docker build -t iris-ml-api:v1.0 .

# Run container
docker run -d -p 5000:5000 --name iris-api iris-ml-api:v1.0

# Test the API
curl -X POST http://localhost:5000/predict \\
  -H "Content-Type: application/json" \\
  -d '{"features": [5.1, 3.5, 1.4, 0.2]}'

# Check health
curl http://localhost:5000/health

# View logs
docker logs iris-api

# Stop container
docker stop iris-api

# Push to Docker Hub
docker tag iris-ml-api:v1.0 yourusername/iris-ml-api:v1.0
docker push yourusername/iris-ml-api:v1.0
```

### Step 5: Test API Locally (Simulated)

In [14]:
from flask import Flask, request, jsonify
import numpy as np

# Simulate API testing
def test_api_endpoints():
    """
    Simulate testing API endpoints
    """
    print(" Testing API Endpoints\n")

    # Test 1: Health check
    print("Test 1: Health Check")
    print("GET /health")
    print('Response: {"status": "healthy", "version": "v1.0.0"}')
    print("✅ Passed\n")

    # Test 2: Prediction
    print("Test 2: Prediction")
    print("POST /predict")
    print('Body: {"features": [5.1, 3.5, 1.4, 0.2]}')

    # Make actual prediction
    sample = np.array([[5.1, 3.5, 1.4, 0.2]])
    pred = model.predict(sample)
    prob = model.predict_proba(sample)

    print(f'Response: {{')
    print(f'  "prediction": {int(pred[0])}')
    print(f'  "probabilities": {prob[0].tolist()}')
    print(f'  "model_version": "v1.0.0"')
    print(f'}}')
    print("✅ Passed\n")

    # Test 3: Model info
    print("Test 3: Model Info")
    print("GET /info")
    print(f'Response: {json.dumps(model_metadata, indent=2)}')
    print("✅ Passed\n")

    print("🎉 All API tests passed!")

test_api_endpoints()

 Testing API Endpoints

Test 1: Health Check
GET /health
Response: {"status": "healthy", "version": "v1.0.0"}
✅ Passed

Test 2: Prediction
POST /predict
Body: {"features": [5.1, 3.5, 1.4, 0.2]}
Response: {
  "prediction": 0
  "probabilities": [1.0, 0.0, 0.0]
  "model_version": "v1.0.0"
}
✅ Passed

Test 3: Model Info
GET /info
Response: {
  "model_name": "iris_classifier",
  "version": "v1.0.0",
  "timestamp": "2026-01-07T16:00:18.592971",
  "accuracy": 1.0,
  "hyperparameters": {
    "n_estimators": 100,
    "random_state": 42
  },
  "training_samples": 120,
  "features": [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)"
  ]
}
✅ Passed

🎉 All API tests passed!


---

## Part 3: Model Drift Detection & Monitoring

### Types of Drift

1. **Data Drift (Covariate Shift):** Input features distribution changes
2. **Concept Drift:** Relationship between features and target changes
3. **Prediction Drift:** Model output distribution changes

In [15]:
# Install drift detection libraries
!pip install -q evidently scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.8/237.8 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.7/565.7 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.7/517.7 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.2/62.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 125.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 456.8/456.8 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 97.4 MB/s eta 0:00:00


### Step 1: Simulate Production Data with Drift

In [16]:
from scipy import stats
import matplotlib.pyplot as plt

In [17]:
# Original training data statistics
print(" Training Data Statistics:")
print(f"Feature 0 mean: {X_train[:, 0].mean():.3f}, std: {X_train[:, 0].std():.3f}")
print(f"Feature 1 mean: {X_train[:, 1].mean():.3f}, std: {X_train[:, 1].std():.3f}\n")

 Training Data Statistics:
Feature 0 mean: 5.809, std: 0.820
Feature 1 mean: 3.062, std: 0.447



In [ ]:
# Simulate production data with drift
np.random.seed(123)


In [18]:
# No drift (healthy production data)
X_prod_nodrift = X_test.copy()

In [20]:
# With data drift (shifted distribution)
X_prod_drift = X_test.copy()
X_prod_drift[:, 0] = X_prod_drift[:, 0] + 0.5  # Shift feature 0
X_prod_drift[:, 1] = X_prod_drift[:, 1] * 1.3  # Scale feature 1

In [ ]:
print(" Production Data (with drift) Statistics:")
print(f"Feature 0 mean: {X_prod_drift[:, 0].mean():.3f}, std: {X_prod_drift[:, 0].std():.3f}")
print(f"Feature 1 mean: {X_prod_drift[:, 1].mean():.3f}, std: {X_prod_drift[:, 1].std():.3f}")

### Step 2: Statistical Drift Detection

In [21]:
from scipy.stats import ks_2samp, wasserstein_distance

def detect_data_drift(reference_data, production_data, feature_names, threshold=0.05):
    """
    Detect data drift using Kolmogorov-Smirnov test
    """
    drift_report = []

    print("🔍 Data Drift Detection Report\n")
    print(f"{'Feature':<20} {'KS Statistic':<15} {'P-Value':<15} {'Drift Detected':<15}")
    print("="*65)

    for i, feature_name in enumerate(feature_names):
        # Kolmogorov-Smirnov test
        ks_stat, p_value = ks_2samp(reference_data[:, i], production_data[:, i])

        # Wasserstein distance
        w_distance = wasserstein_distance(reference_data[:, i], production_data[:, i])

        drift_detected = "⚠️ YES" if p_value < threshold else "✅ NO"

        print(f"{feature_name:<20} {ks_stat:<15.4f} {p_value:<15.4f} {drift_detected:<15}")

        drift_report.append({
            "feature": feature_name,
            "ks_statistic": ks_stat,
            "p_value": p_value,
            "wasserstein_distance": w_distance,
            "drift_detected": p_value < threshold
        })

    return drift_report



KS only tells if drift exists

Wasserstein tells how much drift exists

Example:

p-value small → drift detected

Wasserstein small → drift is minor (may ignore)

Wasserstein large → serious distribution shift

In [22]:
# Test on data without drift
print("Scenario 1: Production Data WITHOUT Drift")
report_no_drift = detect_data_drift(X_train, X_prod_nodrift, iris.feature_names)

# Test on data with drift
print("Scenario 2: Production Data WITH Drift")

report_with_drift = detect_data_drift(X_train, X_prod_drift, iris.feature_names)

Scenario 1: Production Data WITHOUT Drift
🔍 Data Drift Detection Report

Feature              KS Statistic    P-Value         Drift Detected 
sepal length (cm)    0.1167          0.8799          ✅ NO           
sepal width (cm)     0.0667          0.9998          ✅ NO           
petal length (cm)    0.1167          0.8799          ✅ NO           
petal width (cm)     0.1333          0.7603          ✅ NO           
Scenario 2: Production Data WITH Drift
🔍 Data Drift Detection Report

Feature              KS Statistic    P-Value         Drift Detected 
sepal length (cm)    0.3167          0.0137          ⚠️ YES         
sepal width (cm)     0.7333          0.0000          ⚠️ YES         
petal length (cm)    0.1167          0.8799          ✅ NO           
petal width (cm)     0.1333          0.7603          ✅ NO           


Model Drift and Monitoring
## Statistical Drift Detection

Code Explanation: Models decay over time because the world changes.

This code simulates "Production Data" that has drifted (the numbers have shifted or scaled). We use the Kolmogorov-Smirnov (KS) Test.

 It compares the distribution of your training data vs. your live data.

  If the "P-Value" is very small (usually < 0.05), it means the data looks significantly different than what the model was trained on.

### Step 3: Model Performance Drift

In [23]:
def monitor_model_performance(model, X_ref, y_ref, X_prod, y_prod, threshold=0.05):
    """
    Monitor model performance degradation
    """
    # Reference performance
    ref_pred = model.predict(X_ref)
    ref_accuracy = accuracy_score(y_ref, ref_pred)

    # Production performance
    prod_pred = model.predict(X_prod)
    prod_accuracy = accuracy_score(y_prod, prod_pred)

    # Calculate degradation
    degradation = ref_accuracy - prod_accuracy
    degradation_pct = (degradation / ref_accuracy) * 100

    print("📉 Model Performance Monitoring\n")
    print(f"Reference Accuracy (Training): {ref_accuracy:.4f}")
    print(f"Production Accuracy:            {prod_accuracy:.4f}")
    print(f"Performance Degradation:        {degradation:.4f} ({degradation_pct:.2f}%)\n")

    if abs(degradation) > threshold:
        print("⚠️  ALERT: Significant performance degradation detected!")
        print("    Action Required: Retrain model with recent data")
    else:
        print("✅ Model performance is stable")

    return {
        "ref_accuracy": ref_accuracy,
        "prod_accuracy": prod_accuracy,
        "degradation": degradation,
        "alert": abs(degradation) > threshold
    }


In [24]:
# Monitor on data without drift
print("Scenario 1: Monitoring WITHOUT Drift")

perf_no_drift = monitor_model_performance(model, X_train, y_train, X_prod_nodrift, y_test)

# Monitor on data with drift
print("Scenario 2: Monitoring WITH Drift")
y_prod_drift = y_test.copy()  # Assume labels available for monitoring
perf_with_drift = monitor_model_performance(model, X_train, y_train, X_prod_drift, y_prod_drift)

Scenario 1: Monitoring WITHOUT Drift
📉 Model Performance Monitoring

Reference Accuracy (Training): 1.0000
Production Accuracy:            1.0000
Performance Degradation:        0.0000 (0.00%)

✅ Model performance is stable
Scenario 2: Monitoring WITH Drift
📉 Model Performance Monitoring

Reference Accuracy (Training): 1.0000
Production Accuracy:            1.0000
Performance Degradation:        0.0000 (0.00%)

✅ Model performance is stable


### Step 4: Comprehensive Monitoring Dashboard

In [25]:
def create_monitoring_dashboard(model, reference_data, production_data, y_ref, y_prod, feature_names):
    """
    Create comprehensive monitoring dashboard
    """

    print("📊 COMPREHENSIVE MLOps MONITORING DASHBOARD")


    # 1. Data Drift Status
    print("1️⃣  DATA DRIFT STATUS")

    drift_report = detect_data_drift(reference_data, production_data, feature_names, threshold=0.05)
    drift_count = sum([d["drift_detected"] for d in drift_report])
    print(f"\n📈 Summary: {drift_count}/{len(feature_names)} features showing drift\n")

    # 2. Model Performance
    print("\n2️⃣  MODEL PERFORMANCE")

    perf_metrics = monitor_model_performance(model, reference_data, y_ref, production_data, y_prod)

    # 3. Prediction Distribution
    print("\n3️⃣  PREDICTION DISTRIBUTION")

    ref_pred = model.predict(reference_data)
    prod_pred = model.predict(production_data)

    print("Reference Predictions Distribution:")
    for cls in np.unique(y_ref):
        count = np.sum(ref_pred == cls)
        pct = (count / len(ref_pred)) * 100
        print(f"  Class {cls}: {count} ({pct:.1f}%)")

    print("\nProduction Predictions Distribution:")
    for cls in np.unique(y_prod):
        count = np.sum(prod_pred == cls)
        pct = (count / len(prod_pred)) * 100
        print(f"  Class {cls}: {count} ({pct:.1f}%)")

    # 4. Alert Summary
    print("\n4️⃣  ALERT SUMMARY")


    alerts = []
    if drift_count > 0:
        alerts.append(f"⚠️  Data drift detected in {drift_count} feature(s)")
    if perf_metrics["alert"]:
        alerts.append(f"⚠️  Model performance degraded by {abs(perf_metrics['degradation']):.2%}")

    if alerts:
        for alert in alerts:
            print(alert)
        print("\n🔧 Recommended Actions:")
        print("   - Investigate root cause of drift")
        print("   - Collect recent labeled data")
        print("   - Retrain model with updated dataset")
        print("   - Update feature engineering pipeline if needed")
    else:
        print("✅ No alerts - System operating normally")



In [26]:
# Create dashboard for drifted data
create_monitoring_dashboard(model, X_train, X_prod_drift, y_train, y_prod_drift, iris.feature_names)

📊 COMPREHENSIVE MLOps MONITORING DASHBOARD
1️⃣  DATA DRIFT STATUS
🔍 Data Drift Detection Report

Feature              KS Statistic    P-Value         Drift Detected 
sepal length (cm)    0.3167          0.0137          ⚠️ YES         
sepal width (cm)     0.7333          0.0000          ⚠️ YES         
petal length (cm)    0.1167          0.8799          ✅ NO           
petal width (cm)     0.1333          0.7603          ✅ NO           

📈 Summary: 2/4 features showing drift


2️⃣  MODEL PERFORMANCE
📉 Model Performance Monitoring

Reference Accuracy (Training): 1.0000
Production Accuracy:            1.0000
Performance Degradation:        0.0000 (0.00%)

✅ Model performance is stable

3️⃣  PREDICTION DISTRIBUTION
Reference Predictions Distribution:
  Class 0: 40 (33.3%)
  Class 1: 41 (34.2%)
  Class 2: 39 (32.5%)

Production Predictions Distribution:
  Class 0: 10 (33.3%)
  Class 1: 9 (30.0%)
  Class 2: 11 (36.7%)

4️⃣  ALERT SUMMARY
⚠️  Data drift detected in 2 feature(s)

🔧 Recommend

### Step 5: Automated Retraining Pipeline

In [27]:
def automated_retraining_pipeline(current_model, X_train_old, y_train_old, X_new, y_new, drift_threshold=0.05):
    """
    Automated retraining when drift is detected
    """
    print("🔄 AUTOMATED RETRAINING PIPELINE\n")

    # Step 1: Detect drift
    print("Step 1: Drift Detection")
    drift_report = detect_data_drift(X_train_old, X_new, iris.feature_names, threshold=drift_threshold)
    drift_detected = any([d["drift_detected"] for d in drift_report])

    if not drift_detected:
        print("✅ No significant drift detected. Retraining not required.\n")
        return current_model, False

    print("⚠️  Drift detected! Initiating retraining...\n")

    # Step 2: Combine old and new data
    print("Step 2: Data Preparation")
    X_combined = np.vstack([X_train_old, X_new])
    y_combined = np.concatenate([y_train_old, y_new])
    print(f"  Combined dataset size: {len(X_combined)} samples\n")

    # Step 3: Retrain model
    print("Step 3: Model Retraining")
    new_model = RandomForestClassifier(n_estimators=100, random_state=42)
    new_model.fit(X_combined, y_combined)
    print("  ✅ Model retrained\n")

    # Step 4: Validate new model
    print("Step 4: Model Validation")
    old_accuracy = accuracy_score(y_new, current_model.predict(X_new))
    new_accuracy = accuracy_score(y_new, new_model.predict(X_new))

    print(f"  Old model accuracy: {old_accuracy:.4f}")
    print(f"  New model accuracy: {new_accuracy:.4f}")

    if new_accuracy >= old_accuracy:
        print(f"  ✅ New model performs better (Δ = +{new_accuracy - old_accuracy:.4f})\n")

        # Step 5: Deploy new model
        print("Step 5: Model Deployment")
        print("  - Saving new model as v2.0.0")
        print("  - Updating model registry")
        print("  - Triggering CI/CD pipeline")
        print("  ✅ Deployment complete\n")

        return new_model, True
    else:
        print(f"  ⚠️  New model performs worse (Δ = {new_accuracy - old_accuracy:.4f})")
        print("  ❌ Deployment cancelled. Keeping current model.\n")
        return current_model, False



In [28]:
# Simulate automated retraining

new_model, retrained = automated_retraining_pipeline(
    model, X_train, y_train, X_prod_drift, y_prod_drift
)

🔄 AUTOMATED RETRAINING PIPELINE

Step 1: Drift Detection
🔍 Data Drift Detection Report

Feature              KS Statistic    P-Value         Drift Detected 
sepal length (cm)    0.3167          0.0137          ⚠️ YES         
sepal width (cm)     0.7333          0.0000          ⚠️ YES         
petal length (cm)    0.1167          0.8799          ✅ NO           
petal width (cm)     0.1333          0.7603          ✅ NO           
⚠️  Drift detected! Initiating retraining...

Step 2: Data Preparation
  Combined dataset size: 150 samples

Step 3: Model Retraining
  ✅ Model retrained

Step 4: Model Validation
  Old model accuracy: 1.0000
  New model accuracy: 1.0000
  ✅ New model performs better (Δ = +0.0000)

Step 5: Model Deployment
  - Saving new model as v2.0.0
  - Updating model registry
  - Triggering CI/CD pipeline
  ✅ Deployment complete



## Automated Retraining Pipeline
Code Explanation: This is the "Auto-pilot" of MLOps. If the monitoring dashboard detects drift, this function:

Combines the old training data with the new drifted data.

Trains a "V2" model.

Compares V1 and V2 on the new data.

Only deploys V2 if it actually performs better.

##  Hands-On Exercises

### Exercise 1: Extend CI/CD Pipeline
Add these tests to the CI pipeline:
- Test for model inference latency
- Test for memory usage
- Test for bias in predictions

### Exercise 2: Docker Optimization
- Implement multi-stage Docker build
- Add docker-compose.yml for multi-container setup
- Implement logging to external service

### Exercise 3: Advanced Monitoring
- Implement concept drift detection
- Create email alerts for drift detection
- Build a simple dashboard with Streamlit

### Exercise 4: Real-World Scenario
Deploy a different model (e.g., regression, NLP) following the same MLOps principles


